In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df=pd.read_csv("RFM_Raw.csv")
df.head()

,Date,InvoNo,CustomerKey,Amount
0,2018/01/01,JA1803995,A0620,411.400
1,2018/01/01,JA1805774,A1336,658.240
2,2018/01/01,JA1801197,A0786,1892.352
3,2018/01/02,JA1804194,A1089,102.850
4,2018/01/02,JA1805574,A1055,205.700


In [6]:
# 假設你的原始 DataFrame 叫做 df
# 欄位名稱分別為：CustomerKey、Date、InvoNo、Amount

# 1. 確保Date欄位是 datetime 型態，以便進行Date計算
df['Date'] = pd.to_datetime(df['Date'])

# 2. 找出整份資料集中的最大Date（作為基準點來計算 R）
max_date = df['Date'].max()

# 3. 使用 groupby 進行聚合運算
result_df = (
    df.groupby('CustomerKey')
    .agg(
        F=('InvoNo', 'count'),  # InvoNo計數
        M=('Amount', 'sum'),  # Amount加總
        max_date=('Date', 'max'),  # 先保留該客戶的最後交易日
    )
    .reset_index()
)

# 4. 計算 R：基準最大Date 與 該客戶最大Date 的差額
# .dt.days 可以將時間差（Timedelta）轉換為整數天數
result_df['R'] = (max_date - result_df['max_date']).dt.days

# 5. 重新命名欄位並調整順序，符合你的要求：Customer, F, M, R
result_df = result_df.rename(columns={'CustomerKey': 'Customer'})[
    ['Customer', 'F', 'M', 'R']
]

# 顯示結果
print(result_df)

    Customer   F            M   R
0      A0350  25  51238.11440   9
1      A0351   6  17890.31200  16
2      A0352  14  46424.03852  14
3      A0353  14  27607.68636  24
4      A0354   6   9973.92064  31
..       ...  ..          ...  ..
727    A1413  23  52738.79456  10
728    A1415  20  53019.49664  17
729    A1416  24  49938.38378   4
730    A1419  22  64257.84520  50
731    A1420  14  25054.18342  85

[732 rows x 4 columns]


In [10]:
result_df.to_csv("RFM_Data.csv",index=False)